In [2]:
import sys
!{sys.executable} -m pip install pdfplumber chromadb sentence-transformers

  Using cached pdfplumber-0.11.10-py3-none-any.whl.metadata (43 kB)
  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached sentence_transformers-6.0.0-py3-none-any.whl.metadata (20 kB)
  Using cached pdfminer_six-20260107-py3-none-any.whl.metadata (4.3 kB)
  Using cached pillow-12.3.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (9.1 kB)
  Using cached pypdfium2-5.13.0-py3-none-macosx_13_0_arm64.whl.metadata (66 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pybase64-1.5.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (11 kB)
  Using cached uvicorn-0.52.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached onnxruntime-1.29.0-cp313-cp313-macosx_14_0_arm64.whl.metadata (5.6 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl.metadata (1.6 kB)
  

In [4]:
import pdfplumber

pdf = pdfplumber.open("Road-User-Handbook-English.pdf")
print("pages:", len(pdf.pages))

page = pdf.pages[20]
print(page.extract_text()[:1000])

pages: 212
Licences
Licence restrictions
There are restrictions you must follow when you drive with a:
• Learner licence
• Provisional P1 licence (red P)
• Provisional P2 licence (green P).
These restrictions are in place to help keep you safe as you develop your
driving skills. They also apply when you drive in another state or territory.
If you do not follow these restrictions, you’ll get a fine. You can also get
demerit points or even lose your licence.
Restriction Applies to
Alcohol limit 90
Your blood alcohol concentration (BAC) must be zero.
This means you must not have any alcohol in your system
when you drive.
Maximum speed limit 90
You can drive to a maximum of 90km/h.
You must observe all speed limits below 90km/h.
You can drive to a maximum of 100km/h.
You must observe all speed limits below 100km/h.
Road User Handbook 1199


In [5]:
import re

def clean_text(text):
    if not text:
        return ""
    lines = []
    for line in text.split("\n"):
        s = line.strip()
        if not s:
            continue
        if re.fullmatch(r"\d{1,3}", s):     # bare page numbers
            continue
        if re.search(r"road user handbook", s, re.I):   # repeated header
            continue
        lines.append(s)
    return " ".join(lines)

print(clean_text(pdf.pages[20].extract_text())[:600])

Licences Licence restrictions There are restrictions you must follow when you drive with a: • Learner licence • Provisional P1 licence (red P) • Provisional P2 licence (green P). These restrictions are in place to help keep you safe as you develop your driving skills. They also apply when you drive in another state or territory. If you do not follow these restrictions, you’ll get a fine. You can also get demerit points or even lose your licence. Restriction Applies to Alcohol limit 90 Your blood alcohol concentration (BAC) must be zero. This means you must not have any alcohol in your system w


In [6]:
pages = []
for i, page in enumerate(pdf.pages, start=1):
    cleaned = clean_text(page.extract_text())
    if len(cleaned.split()) < 30:     # skip near-empty / image-only pages
        continue
    pages.append({"page": i, "text": cleaned})

print(f"kept {len(pages)} of {len(pdf.pages)} pages")
print(pages[10])

kept 208 of 212 pages
{'page': 13, 'text': 'Licences Step 2: Learner licence Once you’ve passed the DKT and have your learner licence, you can start practising driving (with a supervisor who has a full Australian driver licence). • If you’re over 25, you can go to step 3 as soon as you feel ready. You do not need to log any driving hours. • If you’re under 25, you need to start logging hours. Your learner licence is valid for 5 years but if you let it expire. you will need to pass another DKT and pay the test and licence fee again. Logging driving hours You need to log at least 120 hours of driving, including 20 hours at night, before you can take the Driving Test. To help you gain experience and log your hours, you can: • Do the Safer Drivers Course. This will help you develop hazard perception and safe driving skills. You get 20 hours log book credit once you complete the course. • Have structured ‘3 for 1’ driving lessons with a licensed driving instructor. For every 1 hour lesson, 

In [7]:
import json, os
os.makedirs("data/processed", exist_ok=True)
with open("data/processed/handbook_pages.jsonl", "w") as f:
    for p in pages:
        f.write(json.dumps(p) + "\n")

In [8]:
def chunk_pages(pages, size_words=375, overlap_words=40):
    chunks = []
    cid = 0
    for p in pages:
        words = p["text"].split()
        start = 0
        while start < len(words):
            window = words[start:start + size_words]
            chunks.append({
                "chunk_id": f"c{cid:04d}",
                "page": p["page"],
                "text": " ".join(window),
            })
            cid += 1
            if start + size_words >= len(words):
                break
            start += (size_words - overlap_words)
    return chunks

chunks = chunk_pages(pages)
print(f"{len(chunks)} chunks")
print(chunks[0]["text"][:300])

208 chunks
Introduction to drive legally and safely on NSW roads. It’s essential reading for anyone applying for a driver licence or learning to drive. This handbook is written in plain, easy to understand language, so it is not a detailed version of road law. To access the current and complete set of Road Rul


In [9]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="data/chroma")

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
col = client.create_collection(name="handbook", embedding_function=ef)

col.add(
    ids=[c["chunk_id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"page": c["page"]} for c in chunks],
)
print("indexed:", col.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

indexed: 208


In [10]:
try:
    client.delete_collection("handbook")
except Exception:
    pass

In [11]:
res = col.query(query_texts=["blood alcohol limit for learner drivers"], n_results=3)
for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
    print(f"[p.{meta['page']}] {doc[:200]}\n")

NotFoundError: Error getting collection: Collection [7ecdbe58-3c69-41d4-9e77-e6d3a5af521c] does not exist.

In [12]:
col = client.get_collection("handbook", embedding_function=ef)
print("count:", col.count())

NotFoundError: Collection [handbook] does not exist

In [13]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="data/chroma")

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# re-chunk smaller first
chunks = chunk_pages(pages, size_words=150, overlap_words=30)
print(f"{len(chunks)} chunks")

try:
    client.delete_collection("handbook")
except Exception:
    pass

col = client.create_collection(name="handbook", embedding_function=ef)

col.add(
    ids=[c["chunk_id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"page": c["page"]} for c in chunks],
)

print("indexed:", col.count())

348 chunks
indexed: 348


In [14]:
res = col.query(query_texts=["blood alcohol limit for learner drivers"], n_results=3)
for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
    print(f"[p.{meta['page']}] {doc[:200]}\n")

[p.31] Licences Alcohol and drugs When supervising a learning driver, a supervisor must: • have a blood alcohol concentration (BAC) under 0.05 • not have illegal drugs present in their system or be under the

[p.41] Safe driving behaviour Drive to road conditions Even if you’re driving at or below the speed limit, you may be driving too fast for road conditions such as curves, rain, heavy traffic or night-time. A

[p.199] driving over the alcohol limit (under 0.08, first offence) • not paying a fine • being medically unfit to drive. For some offences, police can suspend and take away your licence immediately. These inc



In [15]:
hits = [c for c in chunks if "must be zero" in c["text"]]
print(len(hits))
for h in hits:
    print(f"[{h['chunk_id']}] p.{h['page']}: {h['text'][:250]}\n")

1
[c0031] p.21: Licences Licence restrictions There are restrictions you must follow when you drive with a: • Learner licence • Provisional P1 licence (red P) • Provisional P2 licence (green P). These restrictions are in place to help keep you safe as you develop yo



In [16]:
res = col.query(query_texts=["blood alcohol limit for learner drivers"], n_results=10)
for rank, (cid, meta) in enumerate(zip(res["ids"][0], res["metadatas"][0]), start=1):
    marker = "  <-- TARGET" if cid == "c0031" else ""
    print(f"{rank}. {cid} p.{meta['page']}{marker}")

1. c0048 p.31
2. c0065 p.41
3. c0324 p.199
4. c0327 p.201
5. c0031 p.21  <-- TARGET
6. c0067 p.42
7. c0069 p.43
8. c0326 p.200
9. c0047 p.30
10. c0051 p.33


In [17]:
res = col.query(query_texts=["..."], n_results=5)